# 1. 导入相关包

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# 2. RMSNorm

In [ ]:
class RMSNorm(nn.Module):
    """
    RMSNorm (Root Mean Square Layer Normalization)
    相比 LayerNorm，去掉了均值中心化，只保留方差归一化，计算更高效。
    LLaMA 和 Qwen 均采用此结构。
    """
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        # 可学习的缩放参数
        self.weight = nn.Parameter(torch.ones(dim))

    def _norm(self, x):
        # 计算均方根：sqrt(mean(x^2))
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)

    def forward(self, x):
        # 转换为 float32 计算以保证精度，最后转回原数据类型
        output = self._norm(x.float()).type_as(x)
        return output * self.weight

# 3. RoPE

In [ ]:
class RotaryEmbedding(nn.Module):
    """
    RoPE (Rotary Positional Embedding)
    通过旋转矩阵将位置信息注入到 Q 和 K 中。
    """
    def __init__(self, dim, max_seq_len=2048, base=10000):
        super().__init__()
        self.dim = dim
        self.max_seq_len = max_seq_len
        self.base = base
        
        # 预计算频率表：1 / (base ^ (2i / dim))
        # 只计算一半维度，因为 RoPE 是成对旋转的
        inv_freq = 1.0 / (self.base ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq)
        
        # 预计算正弦和余弦表
        self._build_cache()

    def _build_cache(self):
        # 生成位置序列 [0, 1, ..., max_seq_len-1]
        t = torch.arange(self.max_seq_len, device=self.inv_freq.device)
        # 计算外积：[seq_len, 1] * [1, dim/2] -> [seq_len, dim/2]
        freqs = torch.einsum('i,j->ij', t, self.inv_freq)
        # 拼接得到完整的 [seq_len, dim]
        emb = torch.cat((freqs, freqs), dim=-1)
        
        self.register_buffer("cos_cached", emb.cos())
        self.register_buffer("sin_cached", emb.sin())

    def forward(self, x, seq_len=None):
        # x: [batch, heads, seq_len, head_dim]
        if seq_len is None:
            seq_len = x.shape[2]
            
        # 根据当前序列长度切片
        # cos/sin shape: [seq_len, head_dim]
        return (
            self.cos_cached[:seq_len, ...].to(x.device),
            self.sin_cached[:seq_len, ...].to(x.device)
        )

def rotate_half(x):
    """辅助函数：将向量后半部分和前半部分交换并取反"""
    x1, x2 = x[..., : x.shape[-1] // 2], x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)

def apply_rope(q, k, cos, sin):
    """
    应用 RoPE 到 Q 和 K
    公式：q_new = q * cos + rotate_half(q) * sin
    """
    # 增加维度以匹配 [batch, heads, seq_len, head_dim]
    cos = cos[None, None, :, :]
    sin = sin[None, None, :, :]
    
    q_embed = (q * cos) + (rotate_half(q) * sin)
    k_embed = (k * cos) + (rotate_half(k) * sin)
    
    return q_embed, k_embed

# 4. Attention

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.n_heads = config.n_heads
        self.n_kv_heads = config.n_heads # 这里简化为 GQA 的特例 MHA
        self.head_dim = config.dim // config.n_heads
        
        # Q, K, V 的投影层 (无偏置项)
        self.wq = nn.Linear(config.dim, config.dim, bias=False)
        self.wk = nn.Linear(config.dim, config.dim, bias=False)
        self.wv = nn.Linear(config.dim, config.dim, bias=False)
        self.wo = nn.Linear(config.dim, config.dim, bias=False) # 输出投影
        
        self.rope = RotaryEmbedding(dim=self.head_dim)
        self.attn_dropout = nn.Dropout(config.dropout)

    def forward(self, x, mask=None):
        B, T, C = x.size() # Batch, Time, Channel
        
        # 1. 线性投影并重塑多头
        q = self.wq(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2) # [B, H, T, D]
        k = self.wk(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.wv(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        
        # 2. 应用 RoPE
        cos, sin = self.rope(v, seq_len=T)
        q, k = apply_rope(q, k, cos, sin)
        
        # 3. 计算注意力分数
        # (B, H, T, D) @ (B, H, D, T) -> (B, H, T, T)
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        
        # 4. 应用因果掩码 (防止看未来)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        else:
            # 自动生成下三角掩码
            mask = torch.tril(torch.ones(T, T, device=x.device)).view(1, 1, T, T)
            scores = scores.masked_fill(mask == 0, float('-inf'))
            
        # 5. Softmax 和 Dropout
        scores = F.softmax(scores.float(), dim=-1).type_as(q)
        scores = self.attn_dropout(scores)
        
        # 6. 加权求和
        out = torch.matmul(scores, v) # [B, H, T, D]
        
        # 7. 恢复维度并输出投影
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.wo(out)

# 5. FFN

In [ ]:
class FeedForward(nn.Module):
    """
    SwiGLU 前馈网络
    结构：(SiLU(W1*x) * W3*x) * W2
    相比 ReLU，SwiGLU 能保留更多梯度信息。
    """
    def __init__(self, config):
        super().__init__()
        # 隐藏层维度通常是输入维度的 4 倍 (或 8/3 倍)
        hidden_dim = int(4 * config.dim)
        # 确保能被 64 整除以提高 GPU 效率
        hidden_dim = int(2 * hidden_dim / 3)
        hidden_dim = ((hidden_dim + 63) // 64) * 64
        
        self.w1 = nn.Linear(config.dim, hidden_dim, bias=False) # Gate
        self.w3 = nn.Linear(config.dim, hidden_dim, bias=False) # Value
        self.w2 = nn.Linear(hidden_dim, config.dim, bias=False) # Output
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        # SiLU 激活函数
        return self.dropout(self.w2(F.silu(self.w1(x)) * self.w3(x)))

# 6. TransformerBlock

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.attention = CausalSelfAttention(config)
        self.feed_forward = FeedForward(config)
        self.attention_norm = RMSNorm(config.dim, eps=config.norm_eps)
        self.ffn_norm = RMSNorm(config.dim, eps=config.norm_eps)

    def forward(self, x, mask=None):
        # Pre-Norm 结构：先归一化，再计算，最后残差连接
        x = x + self.attention(self.attention_norm(x), mask)
        x = x + self.feed_forward(self.ffn_norm(x))
        return x

# 7. 训练参数

In [ ]:
class MiniGPTConfig:
    def __init__(self, vocab_size=32000, dim=512, n_heads=8, n_layers=8, 
                 max_seq_len=512, dropout=0.1, norm_eps=1e-5):
        self.vocab_size = vocab_size
        self.dim = dim
        self.n_heads = n_heads
        self.n_layers = n_layers
        self.max_seq_len = max_seq_len
        self.dropout = dropout
        self.norm_eps = norm_eps

# 8. 模型定义

In [ ]:
class MiniGPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        
        # 1. 词嵌入层
        self.tok_embeddings = nn.Embedding(config.vocab_size, config.dim)
        self.dropout = nn.Dropout(config.dropout)
        
        # 2. 堆叠 Transformer Block
        self.layers = nn.ModuleList([TransformerBlock(config) for _ in range(config.n_layers)])
        
        # 3. 最终归一化
        self.norm = RMSNorm(config.dim, eps=config.norm_eps)
        
        # 4. 输出层 (LM Head)
        self.output = nn.Linear(config.dim, config.vocab_size, bias=False)
        
        # 权重绑定 (可选优化)：让输出层权重与输入嵌入层共享
        # self.output.weight = self.tok_embeddings.weight

    def forward(self, input_ids, targets=None):
        B, T = input_ids.shape
        assert T <= self.config.max_seq_len, "序列长度超过最大限制"
        
        # 1. 嵌入
        h = self.tok_embeddings(input_ids)
        h = self.dropout(h)
        
        # 2. 通过所有层
        for layer in self.layers:
            h = layer(h)
            
        # 3. 最终归一化
        h = self.norm(h)
        
        # 4. 计算 Logits
        logits = self.output(h)
        
        loss = None
        if targets is not None:
            # 计算交叉熵损失
            # 展平张量以便计算 loss
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)), 
                targets.view(-1), 
                ignore_index=-1
            )
            
        return logits, loss

# 9. 训练模型

In [ ]:
def train_mini_gpt():
    # --- 配置 ---
    config = MiniGPTConfig(vocab_size=1000, dim=256, n_heads=8, n_layers=4, max_seq_len=128)
    model = MiniGPT(config)
    
    # 简单的 AdamW 优化器
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, betas=(0.9, 0.95), weight_decay=0.1)
    
    # --- 模拟数据 ---
    # 假设 batch_size=4, seq_len=32 的随机整数输入
    batch_size = 4
    seq_len = 32
    # 输入是随机 token ID
    inputs = torch.randint(0, config.vocab_size, (batch_size, seq_len))
    # 目标是下一个 token (左移一位)
    targets = inputs.clone()
    targets[:, :-1] = inputs[:, 1:]
    targets[:, -1] = -1 # 最后一个位置不需要预测，设为 -1 (ignore_index)

    print(f"开始训练... 模型参数量: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")
    
    model.train()
    for step in range(100): # 仅演示 100 步
        optimizer.zero_grad()
        
        # 前向传播
        logits, loss = model(inputs, targets)
        
        # 反向传播
        loss.backward()
        
        # 梯度裁剪 (防止梯度爆炸)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        
        optimizer.step()
        
        if step % 20 == 0:
            print(f"Step {step}, Loss: {loss.item():.4f}")
    # --- 简单推理测试 ---
    print("\n--- 推理测试 ---")
    model.eval()
    # 给定一个简单的输入序列
    prompt = torch.tensor([[5, 10, 20]]) # [B, T]
    
    with torch.no_grad():
        logits, _ = model(prompt)
        # 取最后一个位置的预测结果
        next_token_logits = logits[:, -1, :]
        # 贪婪采样：取概率最大的 token
        next_token = torch.argmax(next_token_logits, dim=-1)
        print(f"输入序列: {prompt.tolist()}")
        print(f"预测下一个 Token ID: {next_token.tolist()}")

if __name__ == "__main__":
    train_mini_gpt()
